In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv('../dataGenerated/gurgaon_properties_missing_value_imputation.csv')

In [3]:
df.head()

,property_type,society,sector,price,price_per_sqft,areaWithType,bedRoom,bathroom,balcony,floorNum,facing,built_up_area,study room,servant room,store room,pooja room,others,age_category,furnishing_type,luxury_score,combined_rating
0,flat,maa bhagwati residency,sector 7,0.45,5000.0,Carpet area: 900 (83.61 sq.m.),2,2,1,4.0,West,1000.0,0,0,0,0,0,Relatively New,Unfurnished,11,4.00
1,flat,apna enclave,sector 3,0.50,7692.0,Carpet area: 650 (60.39 sq.m.),2,2,1,1.0,West,722.0,0,0,0,0,0,Old Property,Furnished,14,4.25
2,flat,tulsiani easy in homes,sohna road,0.40,6723.0,Carpet area: 595 (55.28 sq.m.),2,2,3,12.0,East,661.0,0,0,0,0,0,New Property,Unfurnished,31,4.25
3,flat,smart world orchard,sector 61,1.47,12250.0,Carpet area: 1200 (111.48 sq.m.),2,2,2,2.0,East,1333.0,1,0,0,0,0,Relatively New,Unfurnished,49,4.20
4,flat,parkwood westend,sector 92,0.70,5204.0,Super Built up area 1345(124.95 sq.m.),2,2,3,5.0,East,1217.0,1,0,0,0,0,Under Construction,Unfurnished,0,4.00


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3573 entries, 0 to 3572
Data columns (total 21 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   property_type    3573 non-null   object 
 1   society          3573 non-null   object 
 2   sector           3573 non-null   object 
 3   price            3573 non-null   float64
 4   price_per_sqft   3573 non-null   float64
 5   areaWithType     3573 non-null   object 
 6   bedRoom          3573 non-null   int64  
 7   bathroom         3573 non-null   int64  
 8   balcony          3573 non-null   object 
 9   floorNum         3573 non-null   float64
 10  facing           3573 non-null   object 
 11  built_up_area    3573 non-null   float64
 12  study room       3573 non-null   int64  
 13  servant room     3573 non-null   int64  
 14  store room       3573 non-null   int64  
 15  pooja room       3573 non-null   int64  
 16  others           3573 non-null   int64  
 17  age_category  

In [5]:
import requests
import time

def get_gurugram_sector_coordinates(sectors):
    """
    Get latitude and longitude for Gurugram sectors.
    Returns a dictionary: {sector: (latitude, longitude)}
    """

    results = {}

    headers = {
        "User-Agent": "gurugram-sector-geocoder/1.0"
    }

    for sector in sectors:
        query = f"{sector}, Gurugram, Haryana, India"

        url = "https://nominatim.openstreetmap.org/search"
        params = {
            "q": query,
            "format": "json",
            "limit": 1
        }

        try:
            response = requests.get(
                url,
                params=params,
                headers=headers,
                timeout=10
            )
            data = response.json()

            if data:
                results[sector] = (
                    float(data[0]["lat"]),
                    float(data[0]["lon"])
                )
            else:
                results[sector] = (None, None)

        except Exception:
            results[sector] = (None, None)

        time.sleep(1)  # respect Nominatim rate limits

    return results

In [6]:
unique_sectors = df["sector"].dropna().unique()

coordinates = get_gurugram_sector_coordinates(unique_sectors)

df["latitude"] = df["sector"].map(
    lambda x: coordinates.get(x, (None, None))[0]
)

df["longitude"] = df["sector"].map(
    lambda x: coordinates.get(x, (None, None))[1]
)

In [7]:
df.head()

,property_type,society,sector,price,price_per_sqft,areaWithType,bedRoom,bathroom,balcony,floorNum,facing,built_up_area,study room,servant room,store room,pooja room,others,age_category,furnishing_type,luxury_score,combined_rating,latitude,longitude
0,flat,maa bhagwati residency,sector 7,0.45,5000.0,Carpet area: 900 (83.61 sq.m.),2,2,1,4.0,West,1000.0,0,0,0,0,0,Relatively New,Unfurnished,11,4.00,28.466257,77.014265
1,flat,apna enclave,sector 3,0.50,7692.0,Carpet area: 650 (60.39 sq.m.),2,2,1,1.0,West,722.0,0,0,0,0,0,Old Property,Furnished,14,4.25,28.497391,77.020526
2,flat,tulsiani easy in homes,sohna road,0.40,6723.0,Carpet area: 595 (55.28 sq.m.),2,2,3,12.0,East,661.0,0,0,0,0,0,New Property,Unfurnished,31,4.25,28.399920,77.045127
3,flat,smart world orchard,sector 61,1.47,12250.0,Carpet area: 1200 (111.48 sq.m.),2,2,2,2.0,East,1333.0,1,0,0,0,0,Relatively New,Unfurnished,49,4.20,28.411019,77.096368
4,flat,parkwood westend,sector 92,0.70,5204.0,Super Built up area 1345(124.95 sq.m.),2,2,3,5.0,East,1217.0,1,0,0,0,0,Under Construction,Unfurnished,0,4.00,28.408905,76.915523


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3573 entries, 0 to 3572
Data columns (total 23 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   property_type    3573 non-null   object 
 1   society          3573 non-null   object 
 2   sector           3573 non-null   object 
 3   price            3573 non-null   float64
 4   price_per_sqft   3573 non-null   float64
 5   areaWithType     3573 non-null   object 
 6   bedRoom          3573 non-null   int64  
 7   bathroom         3573 non-null   int64  
 8   balcony          3573 non-null   object 
 9   floorNum         3573 non-null   float64
 10  facing           3573 non-null   object 
 11  built_up_area    3573 non-null   float64
 12  study room       3573 non-null   int64  
 13  servant room     3573 non-null   int64  
 14  store room       3573 non-null   int64  
 15  pooja room       3573 non-null   int64  
 16  others           3573 non-null   int64  
 17  age_category  

In [11]:
df['latitude'] = df['latitude'].fillna(28.4947)
df['longitude'] = df['longitude'].fillna(77.0226)

In [14]:
df['latitude'] = df['latitude'].round(4)
df['longitude'] = df['longitude'].round(4)

In [15]:
df.head()

,property_type,society,sector,price,price_per_sqft,areaWithType,bedRoom,bathroom,balcony,floorNum,facing,built_up_area,study room,servant room,store room,pooja room,others,age_category,furnishing_type,luxury_score,combined_rating,latitude,longitude
0,flat,maa bhagwati residency,sector 7,0.45,5000.0,Carpet area: 900 (83.61 sq.m.),2,2,1,4.0,West,1000.0,0,0,0,0,0,Relatively New,Unfurnished,11,4.00,28.4663,77.0143
1,flat,apna enclave,sector 3,0.50,7692.0,Carpet area: 650 (60.39 sq.m.),2,2,1,1.0,West,722.0,0,0,0,0,0,Old Property,Furnished,14,4.25,28.4974,77.0205
2,flat,tulsiani easy in homes,sohna road,0.40,6723.0,Carpet area: 595 (55.28 sq.m.),2,2,3,12.0,East,661.0,0,0,0,0,0,New Property,Unfurnished,31,4.25,28.3999,77.0451
3,flat,smart world orchard,sector 61,1.47,12250.0,Carpet area: 1200 (111.48 sq.m.),2,2,2,2.0,East,1333.0,1,0,0,0,0,Relatively New,Unfurnished,49,4.20,28.4110,77.0964
4,flat,parkwood westend,sector 92,0.70,5204.0,Super Built up area 1345(124.95 sq.m.),2,2,3,5.0,East,1217.0,1,0,0,0,0,Under Construction,Unfurnished,0,4.00,28.4089,76.9155


In [16]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # km

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1-a))

In [17]:
CYBER_CITY = (28.4946, 77.0888)

df["dist_cyber_city_km"] = df.apply(
    lambda x: haversine(
        x["latitude"], x["longitude"],
        CYBER_CITY[0], CYBER_CITY[1]
    ),
    axis=1
)

In [18]:
df.head()

,property_type,society,sector,price,price_per_sqft,areaWithType,bedRoom,bathroom,balcony,floorNum,facing,built_up_area,study room,servant room,store room,pooja room,others,age_category,furnishing_type,luxury_score,combined_rating,latitude,longitude,dist_cyber_city_km
0,flat,maa bhagwati residency,sector 7,0.45,5000.0,Carpet area: 900 (83.61 sq.m.),2,2,1,4.0,West,1000.0,0,0,0,0,0,Relatively New,Unfurnished,11,4.00,28.4663,77.0143,7.932372
1,flat,apna enclave,sector 3,0.50,7692.0,Carpet area: 650 (60.39 sq.m.),2,2,1,1.0,West,722.0,0,0,0,0,0,Old Property,Furnished,14,4.25,28.4974,77.0205,6.681787
2,flat,tulsiani easy in homes,sohna road,0.40,6723.0,Carpet area: 595 (55.28 sq.m.),2,2,3,12.0,East,661.0,0,0,0,0,0,New Property,Unfurnished,31,4.25,28.3999,77.0451,11.363911
3,flat,smart world orchard,sector 61,1.47,12250.0,Carpet area: 1200 (111.48 sq.m.),2,2,2,2.0,East,1333.0,1,0,0,0,0,Relatively New,Unfurnished,49,4.20,28.4110,77.0964,9.325542
4,flat,parkwood westend,sector 92,0.70,5204.0,Super Built up area 1345(124.95 sq.m.),2,2,3,5.0,East,1217.0,1,0,0,0,0,Under Construction,Unfurnished,0,4.00,28.4089,76.9155,19.438662


In [19]:
df.corr(numeric_only=True)['price'].sort_values(ascending=False)

price                 1.000000
price_per_sqft        0.793641
built_up_area         0.724482
bathroom              0.610240
bedRoom               0.572662
longitude             0.419529
servant room          0.395196
pooja room            0.323536
store room            0.307230
combined_rating       0.271825
study room            0.246184
latitude              0.145736
luxury_score          0.073795
others               -0.016226
floorNum             -0.090450
dist_cyber_city_km   -0.404338
Name: price, dtype: float64